In [94]:
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models


In [95]:
# 1.1
df = pd.read_csv('pokemon_all.csv')
df = df.dropna(subset=['Type_2']).copy()
pokemon_folder = 'pokemon_png/pokemon_png/'
df.shape

(350, 23)

In [96]:
# 1.2
pokemon_array = []
valid_indices = []

for idx, row in df.iterrows():
    pokemon_number = int(row['Number'])
    filepath = f'{pokemon_folder}{pokemon_number}.png'

    try:
        with Image.open(filepath) as img:
            gray_img = img.convert('L').resize((256,256))
            arr = np.array(gray_img,dtype=np.float32)
            pokemon_array.append(arr)
            valid_indices.append(idx)
    except (FileNotFoundError, OSError):
        continue
x = np.stack(pokemon_array, axis = 0)
df_match = df.loc[valid_indices].reset_index(drop=True)
print(x.shape)

(342, 256, 256)


In [97]:
# 1.3
min_val = x.min()
max_val = x.max()

x = (x - min_val) / (max_val - min_val)

In [98]:
# Question 1
print(x[0,70,35])

0.7019608


In [99]:
# 1.4
dfy = pd.get_dummies(df_match['Type_2'])
y = dfy.to_numpy()

In [100]:
# Question 2
print(y.shape)

(342, 18)


In [101]:
# 2.1
x_train = np.expand_dims(x,axis = -1)
model = models.Sequential([
    layers.Conv2D(16,(3,3),activation='relu', input_shape=(256,256,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32,(3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D((2,2)),
# 2.2
    layers.Flatten(),
# 2.3
    layers.Dense(64,activation='relu'),
# 2.4
    layers.Dense(y.shape[1],activation='softmax'),])

/opt/anaconda3/envs/anaconda-tensorflow/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [102]:
# Question 3
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_16 (Conv2D)              │ (None, 254, 254, 16)   │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 127, 127, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 125, 125, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 60, 60, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │     3,686,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 18)             │         1,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,710,930 (14.16 MB)

 Trainable params: 3,710,930 (14.16 MB)

 Non-trainable params: 0 (0.00 B)

In [103]:
#2.5
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
train = model.fit(x_train,y,epochs=10,batch_size=32)

Epoch 1/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 244ms/step - accuracy: 0.2310 - loss: 2.7526
Epoch 2/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 238ms/step - accuracy: 0.2690 - loss: 2.4484
Epoch 3/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 237ms/step - accuracy: 0.3889 - loss: 2.0509
Epoch 4/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 231ms/step - accuracy: 0.5906 - loss: 1.3505
Epoch 5/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 255ms/step - accuracy: 0.8801 - loss: 0.4706
Epoch 6/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 253ms/step - accuracy: 0.9620 - loss: 0.1218
Epoch 7/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 270ms/step - accuracy: 0.9971 - loss: 0.0218
Epoch 8/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 5s 465ms/step - accuracy: 1.0000 - loss: 0.0044
Epoch 9/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 567ms/step - accuracy: 1.0000 - loss: 9.0051e-04
Epoch 10/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 5s 444ms/step - accuracy: 1.0000 - loss: 6.1879e-04


In [104]:
# Question 4
#Accuracy score is 1.0000

In [105]:
# 3.1 & 3.2
with Image.open('Pokemon_predict_image.png') as p_img:
    p_gray = p_img.convert('L').resize((256,256))
    p_arr = np.array(p_gray,dtype = np.float32)

In [106]:
# 3.3
p_arr = (p_arr - p_arr.min())/(p_arr.max() - p_arr.min())
p_input = np.expand_dims(p_arr,axis=(0,-1))

In [107]:
# 3.4
predictions = model.predict(p_input)[0]
class_names = list(dfy.columns)
predicted_class_idx = np.argmax(predictions)
predicted_class = class_names[predicted_class_idx]
predicted_probability = predictions[predicted_class_idx]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step


In [108]:
# Question 5
print('Most likely class from image:', predicted_class)
print('Probability of belonging to this class:',predicted_probability * 100,'%')

Most likely class from image: Flying
Probability of belonging to this class: 100.0 %
